# Interactive Chat with AI Agents

This notebook demonstrates how to interact with the ADEPT ecosystem's AI agents using the OpenAI-compatible API.

## Security Architecture

All API requests route through `nginx_proxy` (TLS + JWT authentication).
JupyterLab is isolated on `frontend_net` — it cannot reach internal services directly.

## Features Covered
- Chat completions with the ScientificWorkflowAgent
- Streaming responses
- Session management for conversation history
- Using built-in MCP tools
- Multi-turn conversations

## Prerequisites
- Completed authentication setup (00-setup notebooks)
- Valid authentication token

## Docker Configuration
- JupyterLab service: `deployment/local/docker-compose/docker-compose.core.yaml`
- Rebuild after config changes: `make rebuild-jupyterlab`

## 1. Setup and Authentication

In [ ]:
import os
import json
import openai
import httpx
import asyncio
from typing import List, Dict, Any, Optional
from datetime import datetime
import uuid

# Load authentication from setup notebooks
def load_auth():
    try:
        with open("/tmp/adept_auth.json", "r") as f:
            return json.load(f)
    except FileNotFoundError:
        print("Please run 00-setup/01_authentication.ipynb first")
        return None

def load_config():
    try:
        with open("/tmp/adept_config.json", "r") as f:
            return json.load(f)
    except FileNotFoundError:
        print("Please run 00-setup/00_environment_setup.ipynb first")
        return None

auth = load_auth()
config = load_config()

if auth and config:
    print("Authentication loaded successfully")
    print(f"   User: {auth['username']}")
    API_BASE_URL = config.get('api_base_url', os.getenv('ADEPT_API_BASE_URL', 'https://nginx_proxy/v1'))
    NGINX_URL = config.get('nginx_url', 'https://nginx_proxy')
    print(f"   API: {API_BASE_URL}")
else:
    print("Please complete setup notebooks first")

## 2. Initialize OpenAI Client

In [ ]:
# Initialize OpenAI client — routes through nginx_proxy
client = openai.OpenAI(
    base_url=API_BASE_URL,
    api_key=auth['access_token'],
    http_client=httpx.Client(verify=False)
)

# Model to use
MODEL = "agentic-framework:scientific-agent-v1"

print(f"OpenAI client initialized")
print(f"   Endpoint: {API_BASE_URL}")
print(f"   Model: {MODEL}")
print(f"   Routing: nginx_proxy (TLS + JWT)")

## 3. Simple Chat Completion

In [3]:
# Simple single-turn chat
def chat_simple(prompt: str) -> str:
    """Send a simple chat message and get response."""
    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "user", "content": prompt}
            ],
            temperature=0.7,
            max_tokens=500
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"Error: {str(e)}"

# Test simple chat
response = chat_simple("Hello! What scientific tools do you have available?")
print("Agent Response:")
print("-" * 50)
print(response)

Agent Response:
--------------------------------------------------
I’m your persistent-session scientific workflow assistant, and I’ve got a rich toolbox at my disposal to help with everything from literature and data look-ups to sequence analysis, structure prediction, data processing, and even automated multi-agent workflows. Here’s a high-level overview:

1. Knowledge & Literature  
   • Web searches via a robust API (and a scraping fallback)  
   • UniProt queries for protein annotations  
   • PubChem look-ups for chemical compounds (search by name/ID, retrieve properties)  

2. Sequence & Structural Biology  
   • BLAST (protein BLASTP and nucleotide BLASTN) via Biopython against NCBI databases  
   • Large-scale/custom BLAST pipelines on our HPC server (Nextflow)  
   • AlphaFold structure predictions fetched directly from the EBI server (and cached for later queries)  

3. Data & Document Handling  
   • File upload and processing (CSV, Excel, JSON, PDF, DOCX, text, images)  
 

## 4. Streaming Chat Response

In [4]:
# Streaming chat for real-time responses
def chat_stream(prompt: str):
    """Stream chat response token by token."""
    try:
        stream = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "user", "content": prompt}
            ],
            temperature=0.7,
            stream=True
        )
        
        print("Agent Response (streaming):")
        print("-" * 50)
        
        full_response = ""
        for chunk in stream:
            if chunk.choices[0].delta.content:
                content = chunk.choices[0].delta.content
                print(content, end="", flush=True)
                full_response += content
        
        print("\n" + "-" * 50)
        return full_response
        
    except Exception as e:
        print(f"\nError: {str(e)}")
        return None

# Test streaming
response = chat_stream("Explain what BLAST is used for in bioinformatics.")

Agent Response (streaming):
--------------------------------------------------
In bioinformatics, BLAST (Basic Local Alignment Search Tool) is one of the most widely used methods for comparing biological sequences. Here’s what it does and why it’s so important: 1. Sequence Similarity Search • You supply a query sequence—either DNA/RNA (nucleotide) or protein—and BLAST searches against a database of known sequences. • It identifies regions of high local similarity (“hits”) between your query and database entries. 2. Homology and Functional Inference • By finding sequences that are similar to your query, you can infer evolutionary relationships (homology). • If a query sequence matches a protein of known function, you gain clues about the likely function of your unknown sequence. 3. Speed and Statistical Significance • BLAST uses heuristics (word-based indexing and extension) to be far faster than exhaustive alignment methods. • It reports an E-value for each match—an estimate of how man

## 5. Conversational Chat with Session Management

In [5]:
class ChatSession:
    """Manage a conversation session with history."""
    
    def __init__(self, client, model: str = MODEL):
        self.client = client
        self.model = model
        self.messages = []
        self.session_id = str(uuid.uuid4())[:8]
        print(f"📝 New chat session created: {self.session_id}")
    
    def add_message(self, role: str, content: str):
        """Add a message to conversation history."""
        self.messages.append({"role": role, "content": content})
    
    def chat(self, prompt: str, stream: bool = False) -> str:
        """Send a message and get response."""
        # Add user message
        self.add_message("user", prompt)
        
        try:
            if stream:
                response = self._stream_response()
            else:
                response = self._get_response()
            
            # Add assistant response to history
            self.add_message("assistant", response)
            return response
            
        except Exception as e:
            error_msg = f"Error: {str(e)}"
            print(error_msg)
            return error_msg
    
    def _get_response(self) -> str:
        """Get non-streaming response."""
        response = self.client.chat.completions.create(
            model=self.model,
            messages=self.messages,
            temperature=0.7,
            max_tokens=1000,
            extra_body={"session_id": f"notebook_{self.session_id}"}
        )
        return response.choices[0].message.content
    
    def _stream_response(self) -> str:
        """Get streaming response."""
        stream = self.client.chat.completions.create(
            model=self.model,
            messages=self.messages,
            temperature=0.7,
            stream=True,
            extra_body={"session_id": f"notebook_{self.session_id}"}
        )
        
        full_response = ""
        for chunk in stream:
            if chunk.choices[0].delta.content:
                content = chunk.choices[0].delta.content
                print(content, end="", flush=True)
                full_response += content
        print()  # New line after streaming
        return full_response
    
    def clear(self):
        """Clear conversation history."""
        self.messages = []
        print(f"🗑️ Session {self.session_id} cleared")
    
    def show_history(self):
        """Display conversation history."""
        print(f"\n📜 Conversation History (Session: {self.session_id})")
        print("=" * 60)
        for i, msg in enumerate(self.messages, 1):
            role = msg['role'].upper()
            content = msg['content'][:100] + "..." if len(msg['content']) > 100 else msg['content']
            print(f"{i}. [{role}]: {content}")
        print("=" * 60)

# Create a new session
session = ChatSession(client)
print("✅ Chat session ready!")

📝 New chat session created: 2b51a2e5
✅ Chat session ready!


## 6. Multi-Turn Conversation Example

In [6]:
# Have a multi-turn conversation
print("Starting multi-turn conversation...\n")

# Turn 1
print("USER: What is the chemical formula for caffeine?")
print("\nASSISTANT:")
response = session.chat("What is the chemical formula for caffeine?")
print(response)

print("\n" + "=" * 60 + "\n")

Starting multi-turn conversation...

USER: What is the chemical formula for caffeine?

ASSISTANT:
The molecular (chemical) formula for caffeine is  
C8H10N4O2.




In [7]:
# Turn 2 - Follow-up question
print("USER: Can you search for its properties in PubChem?")
print("\nASSISTANT:")
response = session.chat("Can you search for its properties in PubChem?")
print(response)

USER: Can you search for its properties in PubChem?

ASSISTANT:
I found the PubChem entry for caffeine (CID 2519). Here are the key properties:

• Molecular Formula: C₈H₁₀N₄O₂  
• Molecular Weight: 194.19 g/mol  
• IUPAC Name: 1,3,7-trimethylpurine-2,6-dione  
• InChI: InChI=1S/C8H10N4O2/c1-10-4-9-6-5(10)7(13)12(3)8(14)11(6)2/h4H,1-3H3  
• Synonyms: caffeine; 58-08-2; Guaranine; 1,3,7-Trimethylxanthine; Methyltheobromine  

No canonical or isomeric SMILES were returned in this query, but you can retrieve them directly on the PubChem page for CID 2519 if needed.


In [8]:
# Show conversation history
session.show_history()


📜 Conversation History (Session: 2b51a2e5)
1. [USER]: What is the chemical formula for caffeine?
2. [ASSISTANT]: The molecular (chemical) formula for caffeine is  
C8H10N4O2.
3. [USER]: Can you search for its properties in PubChem?
4. [ASSISTANT]: I found the PubChem entry for caffeine (CID 2519). Here are the key properties:

• Molecular Formula...


## 7. Using Scientific Tools

In [9]:
# Create a new session for tool demonstrations
tool_session = ChatSession(client)

print("🔬 Scientific Tool Demonstrations\n")
print("Example 1: Mathematical Calculation")
print("=" * 40)
response = tool_session.chat("Calculate the factorial of 10")
print(response)

📝 New chat session created: 2949b697
🔬 Scientific Tool Demonstrations

Example 1: Mathematical Calculation
The factorial of 10 is 3,628,800.


In [ ]:
print("\nExample 2: Web Search")
print("=" * 40)
response = tool_session.chat("Search for recent developments in CRISPR gene editing")
print(response)

In [ ]:
print("\nExample 3: Code Execution")
print("=" * 40)
response = tool_session.chat(
    "Write and execute Python code to generate the first 20 Fibonacci numbers"
)
print(response)

## 8. Interactive Chat Interface

In [ ]:
def interactive_chat():
    """Interactive chat interface for Jupyter."""
    session = ChatSession(client)
    
    print("🤖 ADEPT Interactive Chat")
    print("=" * 50)
    print("Commands:")
    print("  /clear - Clear conversation history")
    print("  /history - Show conversation history")
    print("  /exit - End chat session")
    print("  /help - Show available tools")
    print("=" * 50)
    print("\nReady to chat! Type your message below.\n")
    
    while True:
        try:
            # Get user input
            user_input = input("You: ").strip()
            
            if not user_input:
                continue
            
            # Handle commands
            if user_input.lower() == "/exit":
                print("👋 Goodbye!")
                break
            elif user_input.lower() == "/clear":
                session.clear()
                continue
            elif user_input.lower() == "/history":
                session.show_history()
                continue
            elif user_input.lower() == "/help":
                response = session.chat("What tools and capabilities do you have available?")
                print(f"\nAssistant: {response}\n")
                continue
            
            # Send message to agent
            print("\nAssistant: ", end="")
            response = session.chat(user_input, stream=True)
            print()  # New line after response
            
        except KeyboardInterrupt:
            print("\n\n👋 Chat interrupted. Goodbye!")
            break
        except Exception as e:
            print(f"\n❌ Error: {str(e)}\n")

# Uncomment to start interactive chat
# interactive_chat()

## 9. Advanced: Custom System Prompts

In [ ]:
# Chat with custom system prompt
def chat_with_system_prompt(system_prompt: str, user_prompt: str) -> str:
    """Chat with a custom system prompt."""
    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            temperature=0.7,
            max_tokens=500
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"Error: {str(e)}"

# Example: Agent as a chemistry tutor
system_prompt = """
You are a friendly chemistry tutor helping students understand chemical concepts.
Use simple language and provide examples when possible.
If asked to perform calculations or look up data, use your available tools.
"""

user_prompt = "Can you explain what a covalent bond is?"

print("System Prompt:")
print(system_prompt)
print("\nUser: " + user_prompt)
print("\nAssistant:")
response = chat_with_system_prompt(system_prompt, user_prompt)
print(response)

## 10. Summary and Next Steps

In [ ]:
print("=" * 60)
print("CHAT INTERFACE TUTORIAL COMPLETE")
print("=" * 60)

print("\n✅ You've learned how to:")
print("   • Send simple chat messages")
print("   • Stream responses in real-time")
print("   • Manage conversation sessions")
print("   • Have multi-turn conversations")
print("   • Use scientific tools through chat")
print("   • Create custom system prompts")

print("\n📚 Next Steps:")
print("\n1. Continue with End User notebooks:")
print("   • 02_file_operations.ipynb - Upload and process files")
print("   • 03_scientific_workflows.ipynb - Run scientific analyses")
print("   • 04_multi_agent_collaboration.ipynb - Work with multiple agents")

print("\n2. Explore Developer notebooks:")
print("   • Learn to build custom tools and agents")
print("   • Integrate with the SDK")
print("   • Create custom workflows")

print("\n💡 Tips:")
print("   • Use streaming for better user experience")
print("   • Maintain session context for coherent conversations")
print("   • Leverage the agent's tools for scientific tasks")
print("   • Check token expiry and refresh when needed")